In [1]:
# ISPY1 Data Cleaning
import os
import numpy as np
import pandas as pd
import nibabel as nib

from google.colab import drive

drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop"

dce_path = os.path.join(base_path, "dce")
mask_path = os.path.join(base_path, "mask")

metadata_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_spy1_metadata.csv"

Mounted at /content/drive
DCE path: /content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop/dce
Mask path: /content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_ISPY1_min_crop/mask
Metadata path: /content/drive/MyDrive/Breast_Cancer_AI_Project/data/BreastDCEDL_spy1_metadata.csv


In [ ]:
dce_files = sorted([
    f for f in os.listdir(dce_path)
    if f.endswith(".nii.gz")
])

mask_files = sorted([
    f for f in os.listdir(mask_path)
    if f.endswith(".nii.gz")
])

metadata = pd.read_csv(metadata_path)

In [ ]:
# Step 1: Build patient-level cohort
image_patient_ids = sorted({
    f.split("_")[1]
    for f in dce_files
})

image_patient_ids = {
    f"ISPY1_{pid}"
    for pid in image_patient_ids
}

In [ ]:
# Step 2: Create patient-level master table
patient_records = []

for patient_id in sorted(image_patient_ids):

    patient_dce = sorted([
        f for f in dce_files
        if f"ISPY1_{patient_id.split('_')[1]}_" in f
    ])

    patient_mask = [
        f for f in mask_files
        if f"ISPY1_{patient_id.split('_')[1]}_" in f
    ]

    patient_metadata = metadata[
        metadata["pid"] == patient_id
    ]

    record = {
        "pid": patient_id,
        "dce_0": os.path.join(dce_path, patient_dce[0]),
        "dce_1": os.path.join(dce_path, patient_dce[1]),
        "dce_2": os.path.join(dce_path, patient_dce[2]),
        "mask": os.path.join(mask_path, patient_mask[0])
    }

    if len(patient_metadata) == 1:
        for column in metadata.columns:
            if column != "pid":
                record[column] = patient_metadata.iloc[0][column]

    patient_records.append(record)


patient_df = pd.DataFrame(patient_records)

In [ ]:
# Step 3: Validate tumor masks
mask_quality = []

for _, row in patient_df.iterrows():

    mask_img = nib.load(row["mask"])
    mask_data = mask_img.get_fdata()

    unique_values = np.unique(mask_data)

    tumor_voxels = np.sum(mask_data > 0)

    mask_quality.append({
        "pid": row["pid"],
        "unique_values": unique_values,
        "tumor_voxels": tumor_voxels,
        "empty": tumor_voxels == 0
    })

mask_quality_df = pd.DataFrame(mask_quality)

print("Empty masks:",
      mask_quality_df["empty"].sum())

In [ ]:
# Check whether all masks are binary
non_binary_masks = []

for _, row in patient_df.iterrows():

    mask_img = nib.load(row["mask"])
    mask_data = mask_img.get_fdata()

    unique_values = np.unique(mask_data)

    if not np.all(np.isin(unique_values, [0, 1])):
        non_binary_masks.append(
            (row["pid"], unique_values)
        )

print("Non-binary masks:", len(non_binary_masks))

if non_binary_masks:
    print("\nNon-binary masks:")
    for item in non_binary_masks:
        print(item)
else:
    print("All masks are binary (0 and 1).")

In [ ]:
# Save validated patient-level dataset
cleaned_path = os.path.join(
    "/content/drive/MyDrive/Breast_Cancer_AI_Project/data",
    "cleaned"
)

os.makedirs(cleaned_path, exist_ok=True)

master_table_path = os.path.join(
    cleaned_path,
    "patient_master.csv"
)

patient_df.to_csv(master_table_path, index=False)

In [ ]:
# Check DCE-mask spatial alignment
affine_mismatches = []

for _, row in patient_df.iterrows():

    dce_img = nib.load(row["dce_0"])
    mask_img = nib.load(row["mask"])

    if not np.allclose(dce_img.affine, mask_img.affine):
        affine_mismatches.append(row["pid"])

print("DCE-mask affine mismatches:", len(affine_mismatches))

if affine_mismatches:
    print("\nMismatched patients:")
    print(affine_mismatches)
else:
    print("All DCE volumes and masks are spatially aligned.")

### DCE Acquisition Phases

- `dce_0` → pre-contrast
- `dce_1` → early post-contrast
- `dce_2` → late post-contrast

In [ ]:
from sklearn.model_selection import train_test_split

master_path = "/content/drive/MyDrive/Breast_Cancer_AI_Project/data/cleaned/patient_master.csv"

patient_df = pd.read_csv(master_path)

In [ ]:
labeled_df = patient_df[patient_df["pCR"].notna()].copy()
missing_pcr_df = patient_df[patient_df["pCR"].isna()].copy()

# 70% train / 15% validation / 15% test
train_df, temp_df = train_test_split(
    labeled_df,
    test_size=0.30,
    stratify=labeled_df["pCR"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["pCR"],
    random_state=42
)

In [ ]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Patient with missing pCR goes to training
missing_pcr_df = missing_pcr_df.copy()
missing_pcr_df["split"] = "train"

patient_df = pd.concat(
    [train_df, val_df, test_df, missing_pcr_df],
    axis=0
).sort_index()

In [ ]:
patient_df.to_csv(master_path, index=False)